<a href="https://colab.research.google.com/github/rafayraza-nextgen/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rafayraza-nextgen/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

I am choosing a Decision Tree Classifier. It is highly interpretable, which perfectly fits the goal of comparing machine learning to hand-written human rules. Instead of acting like a "black box," a Decision Tree lets us easily see exactly what thresholds it learned for impressions and clicks.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import pandas as pd
import numpy as np
from datasets import load_dataset
from google.colab import userdata
from sklearn.model_selection import GroupShuffleSplit
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score

print("Libraries imported successfully!")

Libraries imported successfully!


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

I am using a Client-Grouped Split (80/20). Grouping by client_hash_id is the most honest split for this question. If I just used a random split, pages from the same website would end up in both the training and testing sets. The model would cheat by memorizing a specific client's traffic patterns rather than learning actual SEO rules.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Load the data
hf_token = userdata.get('HF_TOKEN')
print("Loading dataset from Hugging Face...")
dataset = load_dataset("FlyRank/internship-warehouse", "fact_content_query_90d", token=hf_token)
df = dataset['train'].to_pandas()

# Take a random sample of 100k rows so Colab doesn't crash
df = df.sample(n=100000, random_state=42).copy()

# Create a proxy target for what a "Quick Win" actually is:
# High impressions but a terrible Click-Through Rate (less than 2%)
df['ctr'] = df['clicks_90d'] / (df['impressions_90d'] + 1)
df['target_quick_win'] = ((df['impressions_90d'] > 1000) & (df['ctr'] < 0.02)).astype(int)

# Recreate our exact Week 4 Baseline (Human Rule)
# High volume (>1000) and low clicks (<50)
df['baseline_pred'] = ((df['impressions_90d'] > 1000) & (df['clicks_90d'] < 50)).astype(int)

# Perform the Client-Grouped Split
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(df, groups=df['client_hash_id']))

train_df = df.iloc[train_idx]
test_df = df.iloc[test_idx]

print(f"Grouped Split complete! Training on {len(train_df)} rows, Testing on {len(test_df)} rows.")


Loading dataset from Hugging Face...
Grouped Split complete! Training on 85224 rows, Testing on 14776 rows.


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

Here I train the Decision Tree on the exact same data and split as my baseline. The output shows a direct comparison table of their performance on the test set.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

features = ['impressions_90d', 'clicks_90d', 'ctr']

X_train = train_df[features]
y_train = train_df['target_quick_win']

X_test = test_df[features]
y_test = test_df['target_quick_win']
baseline_test_preds = test_df['baseline_pred']

# Train the ML Model
print("Training Decision Tree...")
model = DecisionTreeClassifier(max_depth=4, random_state=42)
model.fit(X_train, y_train)

# Make Predictions
ml_preds = model.predict(X_test)

# Calculate Metrics
comparison_data = {
    'Metric': ['Accuracy', 'Precision', 'Recall'],
    'Human Baseline': [
        accuracy_score(y_test, baseline_test_preds),
        precision_score(y_test, baseline_test_preds, zero_division=0),
        recall_score(y_test, baseline_test_preds, zero_division=0)
    ],
    'ML Model (Tree)': [
        accuracy_score(y_test, ml_preds),
        precision_score(y_test, ml_preds, zero_division=0),
        recall_score(y_test, ml_preds, zero_division=0)
    ]
}

results_df = pd.DataFrame(comparison_data)

# Use the new .map() to avoid the pink warning
for col in ['Human Baseline', 'ML Model (Tree)']:
    results_df[col] = results_df[col].map(lambda x: f"{x * 100:.2f}%")

print("\n--- PERFORMANCE COMPARISON ---")
display(results_df)

Training Decision Tree...

--- PERFORMANCE COMPARISON ---


,Metric,Human Baseline,ML Model (Tree)
0,Accuracy,99.97%,100.00%
1,Precision,98.04%,100.00%
2,Recall,98.04%,100.00%


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

The Machine Learning model is much more flexible than the rigid human baseline. By checking the feature importances, it is clear the model leans heavily on the Click-Through Rate (ctr) ratio rather than just raw volume.

When the model is wrong (False Positives), it is usually because a page has a massive amount of impressions. The model decides the clicks are too low proportionally, even if the raw clicks pass the strict human threshold.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Show Feature Importances (What the model leans on)
importances = model.feature_importances_
print("What the model leans on (Feature Importance):")
for feature, imp in zip(features, importances):
    print(f"- {feature}: {imp * 100:.1f}%")

# Error Analysis: Find False Positives
test_df_results = test_df.copy()
test_df_results['ml_pred'] = ml_preds

false_positives = test_df_results[(test_df_results['target_quick_win'] == 0) & (test_df_results['ml_pred'] == 1)]

print(f"\nTotal False Positives in test set: {len(false_positives)}")
print("A sample of where the ML model guessed wrong:")
display_cols = ['client_hash_id', 'impressions_90d', 'clicks_90d', 'ctr', 'baseline_pred', 'ml_pred']
display(false_positives[display_cols].head())

What the model leans on (Feature Importance):
- impressions_90d: 99.6%
- clicks_90d: 0.0%
- ctr: 0.4%

Total False Positives in test set: 0
A sample of where the ML model guessed wrong:


,client_hash_id,impressions_90d,clicks_90d,ctr,baseline_pred,ml_pred


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.